# Movie reviews: negative, meh or positive?

This demo trains a small **decision tree** that reads a movie review and guesses whether it is *negative*, *meh* or *positive*. It only looks at simple clues you can see for yourself: lists of words (like *bad*, *horrible*, *excellent*), words like *not* and *never*, the review's length, words in ALL CAPS, and punctuation (`!`, `?`, `...`).

Run each cell with <kbd>Shift</kbd>+<kbd>Enter</kbd>, starting from the top. The first run takes a while because your browser downloads Python. Your edits are only saved in this browser; use **File → Download** to keep a copy.

In [ ]:
import re
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

pl.Config.set_fmt_str_lengths(2000)  # show whole reviews in tables instead of cutting them off

# Allows pl.read_csv() to download from a URL inside the browser.
import pyodide_http
pyodide_http.patch_all()

## The data

4,500 Amazon reviews of movies and TV shows, 1,500 for each label. The labels come from the star rating: 1–2 stars is *negative*, 3 is *meh*, 4–5 is *positive*. The [dataset README](https://github.com/jackbandy/data-science-fun/tree/main/datasets/movie-reviews) says where they come from.

In [ ]:
url = "https://raw.githubusercontent.com/jackbandy/data-science-fun/main/datasets/movie-reviews/reviews.csv"
reviews = pl.read_csv(url)
reviews["sentiment"].value_counts()

In [ ]:
reviews.filter(pl.col("sentiment") == "negative").sample(5, seed=418)

In [ ]:
reviews.filter(pl.col("sentiment") == "meh").sample(5, seed=418)

In [ ]:
reviews.filter(pl.col("sentiment") == "positive").sample(5, seed=418)

## Clues the tree can use

Edit the three word lists to change what the tree can look for, then re-run this cell and everything below it. A word only counts as a whole word, ignoring case, so `"bad"` matches *Bad* but not *badly*. Phrases like `"not bad"` work too.

Every word is its own feature/clue, and the tree picks the few that help most. The three lists are just for keeping things tidy: the tree doesn't know which list a word came from. Some words don't go where you'd guess. *good* shows up in more meh reviews than positive ones.

In [ ]:
NEGATIVE_WORDS = [
    "bad", "horrible", "terrible", "awful", "worst", "worse", "boring", "waste", "stupid", "dumb",
    "disappointed", "disappointing", "garbage", "crap", "sucks", "ridiculous", "unwatchable", "pointless",
    "poor", "poorly", "mediocre", "lame", "mess", "annoying", "hated", "refund", "returned",
    "not worth", "don't waste", "fell asleep", "turned it off",
]
MEH_WORDS = [
    "ok", "okay", "decent", "average", "but", "however", "though", "slow", "somewhat", "a bit", "some",
    "expected", "could have", "not bad", "pretty good", "not great", "nothing special", "three stars", "3 stars",
]
POSITIVE_WORDS = [
    "good", "great", "excellent", "love", "loved", "loves", "amazing", "best", "wonderful", "perfect",
    "favorite", "enjoyed", "fantastic", "awesome", "brilliant", "beautiful", "classic", "masterpiece", "fun",
    "superb", "outstanding", "well done", "highly recommend", "must see", "five stars", "5 stars",
]

In [ ]:
def make_features(df):
    """One row per review: how many times each clue shows up in its text."""
    text = pl.col("text")
    lower = text.str.to_lowercase().str.replace_all("’", "'", literal=True)  # curly apostrophes count as straight ones
    count = lambda word: lower.str.count_matches(rf"\b{re.escape(word)}\b")
    return df.select(
        *[count(w).alias(w) for w in dict.fromkeys(NEGATIVE_WORDS + MEH_WORDS + POSITIVE_WORDS)],  # each word once
        lower.str.count_matches(r"\b(not|no|never|nothing|none)\b|n't\b").alias("not/no/never"),
        text.str.count_matches(r"\S+").alias("length"),  # in words
        text.str.count_matches(r"\b[A-Z]{2,}\b").alias("CAPS words"),  # words like LOVED or WORST
        text.str.count_matches("!", literal=True).alias("!"),
        text.str.count_matches("?", literal=True).alias("?"),
        text.str.count_matches("...", literal=True).alias("..."),
    )

X = make_features(reviews)
y = reviews["sentiment"]
X.head()

## Train the tree

The tree learns from 80% of the reviews and is tested on the other 20%, which it never saw. With three balanced labels, guessing at random would be right about 33% of the time. `MAX_DEPTH` is the most questions the tree can ask in a row; try 2 or 3 for a smaller tree.

In [ ]:
MAX_DEPTH = 5

X_train, X_test, y_train, y_test = train_test_split(X.to_numpy(), y.to_numpy(), test_size=0.2, random_state=418)
tree = DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_leaf=25, random_state=418)
tree.fit(X_train, y_train)

print(f"accuracy on training reviews: {tree.score(X_train, y_train):.0%}")
print(f"accuracy on test reviews:     {tree.score(X_test, y_test):.0%}")

## Read the tree

Start at the top. Each square is a question about a clue, and **going right always means the review has it**: `great` means the review uses the word *great*, `not/no/never ≥ 2` means it uses words like *not* or *never* at least twice, and `length ≥ 21` means it is at least 21 words long. Going left means it doesn't. Each oval is an answer, with the share of training reviews there that had that label. Color shows which label is most common (red negative, gray meh, green positive), and a darker color means a clearer majority.

When both sides of a question lead to the same answer, the question doesn't change any prediction, so the drawing merges them into one oval. The tree still predicts exactly the same thing.

In [ ]:
# NOTE: merge_same_answers() and draw_tree() were written by an LLM coding system.
import copy

COLORS = {"negative": "#c0392b", "meh": "#7f7f7f", "positive": "#1e8449"}

def merge_same_answers(tree):
    """A copy of the tree where a question whose two sides give the same answer becomes one leaf. Predictions don't change."""
    tree = copy.deepcopy(tree)
    t = tree.tree_
    def is_leaf(node):  # merges bottom-up, so whole same-answer branches fold into one leaf
        if t.children_left[node] == -1:
            return True
        left, right = t.children_left[node], t.children_right[node]
        both_leaves = is_leaf(left) & is_leaf(right)  # & (not and) so both sides get merged first
        if both_leaves and t.value[left][0].argmax() == t.value[right][0].argmax():
            t.children_left[node] = t.children_right[node] = -1
            return True
        return False
    is_leaf(0)
    return tree

def draw_tree(tree, feature_names):
    """sklearn's plot_tree, relabelled as yes/no questions (squares) and answers (ovals), colored by label."""
    tree = merge_same_answers(tree)
    fig, ax = plt.subplots(figsize=(14, 7))
    annotations = plot_tree(tree, feature_names=feature_names, fontsize=11, ax=ax)
    boxes = [a for a in annotations if a.get_bbox_patch()]
    for a in annotations:  # the two arrow labels under the top box
        if a.get_text().strip() in ("True", "False"):
            a.set_text({"True": "no  ", "False": "  yes"}[a.get_text().strip()])
    t = tree.tree_
    order, stack = [], [0]  # plot_tree draws boxes top-down, left branch first
    while stack:
        node = stack.pop()
        order.append(node)
        if t.children_left[node] != -1:
            stack += [t.children_right[node], t.children_left[node]]
    for node, box in zip(order, boxes):
        shares = t.value[node][0] / t.value[node][0].sum()
        label = tree.classes_[shares.argmax()]
        if t.children_left[node] == -1:  # a leaf: the tree's answer
            box.set_text(f"{label.upper()}\n{shares.max():.0%} of {t.n_node_samples[node]}")
            box.get_bbox_patch().set_boxstyle("ellipse", pad=0.15)
        else:
            # Every clue is a count, and sklearn sends count <= cut left, so right means "has it" (or "at least that many")
            name, cut = feature_names[t.feature[node]], t.threshold[node]
            box.set_text((name if cut < 1 else f"{name} ≥ {int(cut) + 1}") + f"\n{t.n_node_samples[node]} reviews")
            box.get_bbox_patch().set_boxstyle("square", pad=0.4)
        strength = (shares.max() - 1 / 3) / (2 / 3)  # 0 for an even three-way split, 1 for all one label
        rgb = [1 - strength * (1 - c) for c in to_rgb(COLORS[label])]
        box.get_bbox_patch().set(facecolor=rgb, edgecolor=COLORS[label])
        box.set_color("white" if sum(rgb) / 3 < 0.55 else "black")
    ax.set_title("Right = the review has it, left = it doesn't", loc="left", fontsize=14)
    plt.show()

draw_tree(tree, X.columns)

## Try your own reviews

Paste in reviews of any movie, new or old, and re-run the cell.

In [ ]:
my_reviews = pl.DataFrame({"text": [
    "Absolutely LOVED it!!! Best movie of the year.",
    "It was ok. Some good parts, but way too long...",
    "Horrible. A total waste of two hours.",
]})

my_reviews.with_columns(prediction=tree.predict(make_features(my_reviews).to_numpy()))

# Simplified: negative or positive

In [ ]:
# TODO